In [13]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def find_project_dir() -> Path:
    """Find the project root from either a notebook or the repository root."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd() / "cafe-revenue-regression"]
    for candidate in candidates:
        if (candidate / "data" / "raw" / "train.csv").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy file train.csv trong thư mục data/raw của dự án.")


PROJECT_DIR = find_project_dir()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data.load import load_raw_data

datasets = load_raw_data(PROJECT_DIR / "data" / "raw")
train = datasets["train"].copy()
stores = datasets["stores"].copy()
weather = datasets["weather"].copy()
holidays = datasets["holidays"].copy()

print(f"Project: {PROJECT_DIR}")
for name, frame in datasets.items():
    print(f"{name:12s}: {frame.shape}")


train        -> Shape: (26632, 4)
stores       -> Shape: (45, 8)
weather      -> Shape: (1593, 4)
holidays     -> Shape: (55, 3)
sample_sub   -> Shape: (3869, 2)
Project: d:\GitHub\ML-Kaggle-Project\cafe-revenue-regression
train       : (26632, 4)
stores      : (45, 8)
weather     : (1593, 4)
holidays    : (55, 3)
sample_sub  : (3869, 2)


In [14]:
missing_rows = []
for name, frame in datasets.items():
    for column in frame.columns:
        missing = int(frame[column].isna().sum())
        missing_rows.append({
            "dataset": name,
            "column": column,
            "dtype": str(frame[column].dtype),
            "missing_count": missing,
            "missing_pct": round(100 * missing / len(frame), 2) if len(frame) else 0,
            "n_unique": int(frame[column].nunique(dropna=True)),
        })

missing_report = pd.DataFrame(missing_rows).sort_values(["missing_count", "dataset"], ascending=[False, True])
display(missing_report)


,dataset,column,dtype,missing_count,missing_pct,n_unique
15,weather,rain_mm,float64,16,1.00,332
8,stores,area_m2,float64,2,4.44,36
16,holidays,date,datetime64[us],0,0.00,53
17,holidays,holiday_name,string,0,0.00,15
18,holidays,type,str,0,0.00,2
19,sample_sub,id,int64,0,0.00,3869
20,sample_sub,revenue,float64,0,0.00,1
4,stores,store_id,category,0,0.00,45
5,stores,city,string,0,0.00,2
6,stores,district,str,0,0.00,19


In [15]:
key_checks = []
for name, frame in datasets.items():
    key_checks.append({"dataset": name, "duplicate_rows": int(frame.duplicated().sum())})

for name, frame, keys in [
    ("train", train, ["date", "store_id"]),
    ("stores", stores, ["store_id"]),
    ("weather", weather, ["date", "city"]),
    ("holidays", holidays, ["date", "holiday_name"]),
]:
    if set(keys).issubset(frame.columns):
        key_checks.append({"dataset": name, "key": " + ".join(keys), "duplicate_key_rows": int(frame.duplicated(keys, keep=False).sum())})

duplicate_report = pd.DataFrame(key_checks).fillna("")
display(duplicate_report)


,dataset,duplicate_rows,key,duplicate_key_rows
0,train,132.0,,
1,stores,0.0,,
2,weather,0.0,,
3,holidays,0.0,,
4,sample_sub,0.0,,
5,train,,date + store_id,264.0
6,stores,,store_id,0.0
7,weather,,date + city,0.0
8,holidays,,date + holiday_name,0.0


In [17]:
range_checks = [
    {"check": "revenue < 0", "count": int(train["revenue"].lt(0).sum())},
    {"check": "revenue == 0", "count": int(train["revenue"].eq(0).sum())},
    {"check": "is_promo not in {0, 1}", "count": int((~train["is_promo"].isin([0, 1])).sum())},
    {"check": "area_m2 <= 0", "count": int(pd.to_numeric(stores["area_m2"], errors="coerce").le(0).sum())},
    {"check": "rain_mm < 0", "count": int(pd.to_numeric(weather["rain_mm"], errors="coerce").lt(0).sum())},
]
display(pd.DataFrame(range_checks))

train_store_ids = set(train["store_id"].dropna())
store_ids = set(stores["store_id"].dropna())
unknown_store_ids = sorted(train_store_ids - store_ids)
print(f"Store ID không tồn tại trong stores.csv: {len(unknown_store_ids)}", unknown_store_ids)

for name, frame in datasets.items():
    if "date" in frame:
        invalid = int(frame["date"].isna().sum())
        print(f"{name}: date thiếu/không hợp lệ={invalid}, từ={frame['date'].min()}, đến={frame['date'].max()}")


,check,count
0,revenue < 0,0
1,revenue == 0,0
2,"is_promo not in {0, 1}",0
3,area_m2 <= 0,0
4,rain_mm < 0,0


Store ID không tồn tại trong stores.csv: 0 []
train: date thiếu/không hợp lệ=0, từ=2024-01-01 00:00:00, đến=2025-12-31 00:00:00
weather: date thiếu/không hợp lệ=0, từ=2024-01-01 00:00:00, đến=2026-03-31 00:00:00
holidays: date thiếu/không hợp lệ=0, từ=2024-01-01 00:00:00, đến=2026-03-08 00:00:00


In [18]:
revenue = train["revenue"].dropna()
q1, q3 = revenue.quantile([.25, .75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outlier_mask = train["revenue"].lt(lower) | train["revenue"].gt(upper)

outlier_report = pd.DataFrame([{
    "q1": q1, "q3": q3, "iqr": iqr, "lower_bound": lower, "upper_bound": upper,
    "outlier_count": int(outlier_mask.sum()), "outlier_pct": round(100 * outlier_mask.mean(), 2),
    "negative_count": int(train["revenue"].lt(0).sum()),
    "zero_count": int(train["revenue"].eq(0).sum()),
}])
display(outlier_report)
display(train.loc[outlier_mask].sort_values("revenue", ascending=False).head(20))


,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct,negative_count,zero_count
0,12.43,24.129999,11.699999,-5.119998,41.679997,422,1.58,0,0


,date,store_id,is_promo,revenue
2139,2024-03-07,S12,1,53600.0
2071,2024-03-05,S12,1,51990.0
2173,2024-03-08,S12,1,45130.0
2880,2024-03-29,S12,1,44280.0
2105,2024-03-06,S12,0,38120.0
2813,2024-03-27,S12,0,35780.0
2373,2024-03-14,S12,0,32380.0
2273,2024-03-11,S12,0,32190.0
2610,2024-03-21,S12,0,32110.0
2579,2024-03-20,S12,0,31670.0


In [19]:
clean_datasets = {name: frame.copy() for name, frame in datasets.items()}
clean_datasets["train"]["is_promo"] = pd.to_numeric(clean_datasets["train"]["is_promo"], errors="coerce").astype("Int8")
clean_datasets["stores"]["area_m2"] = pd.to_numeric(clean_datasets["stores"]["area_m2"], errors="coerce")
clean_datasets["weather"]["rain_mm"] = pd.to_numeric(clean_datasets["weather"]["rain_mm"], errors="coerce")

# B?o c?o g?n ?? notebook/CI c? th? ki?m tra d? d?ng.
quality_summary = pd.DataFrame({
    "metric": ["train_rows", "missing_cells", "duplicate_rows", "duplicate_train_keys", "iqr_outliers", "unknown_store_ids"],
    "value": [
        len(train), int(train.isna().sum().sum()), int(train.duplicated().sum()),
        int(train.duplicated(["date", "store_id"], keep=False).sum()), int(outlier_mask.sum()), len(unknown_store_ids),
    ],
})
display(quality_summary)


,metric,value
0,train_rows,26632
1,missing_cells,0
2,duplicate_rows,132
3,duplicate_train_keys,264
4,iqr_outliers,422
5,unknown_store_ids,0
